# ContractRisk M3 — evaluación sobre el gold

Ejecuta la evaluación completa de M3 sobre las 12 preguntas del gold (`m3-qa-gold-1.0`):

1. Línea base sin RAG (mismo generador Qwen2.5-3B; adaptación de la fila de M2).
2. RAG ingenuo: recuperación densa top-3 (S07).
3. RAG avanzado: híbrido BM25 + denso con RRF y filtro de ID, más reranking con cross-encoder.
4. Sistema final: RAG avanzado + herramienta `diferencia_valores_contratos` (agente de máximo 3 pasos).
5. Dos ablaciones: solo híbrido, y denso + reranking.

Cada respuesta se evalúa con el harness `m3-harness-1.0` (cobertura de hechos, juez, chequeo de dominio y latencia). Luego se calculan las cuatro métricas RAGAS (implementación del laboratorio S10, juez Qwen2.5-3B local) sobre el ingenuo, el avanzado y el final. Las tablas corregidas con `m3-harness-1.1` se obtienen después, sin GPU, con `m3/rescore_m3.py` (ver `RUTA_ENTREGA.md`).

## Cómo ejecutar

- **Requisitos:** Colab con GPU T4 (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*), en una sesión nueva. No requiere claves ni APIs de pago.
- **Orden:** ejecutar las celdas de arriba hacia abajo. Las celdas hacen, en orden:
  1. obtienen el código;
  2. instalan las dependencias fijadas en `m3/requirements-development-v2.txt` y corren las pruebas;
  3. ejecutan la evaluación;
  4. muestran las tablas y las trazas de la herramienta;
  5. empaquetan los resultados.
- **Versiones:** modelos con revisión fija (Qwen2.5-3B-Instruct, MiniLM multilingüe, cross-encoder mMiniLM), semilla 42 y decodificación greedy. Las versiones efectivas de paquetes, GPU y hashes de código, corpus y gold quedan en `metadata.json`.
- **Tiempo estimado de ejecución:** 30–45 minutos. Al finalizar, se genera un ZIP con los resultados y metadatos de la evaluación.

**Nota metodológica:** las respuestas evaluadas no deben editarse manualmente, y los errores de ejecución o de generación se conservan, para asegurar la trazabilidad de los resultados. Cada ejecución escribe en una carpeta nueva y no sobrescribe resultados anteriores.

In [ ]:
import os, subprocess, sys, tempfile, zipfile
from pathlib import Path
# Obtiene el código: clona la rama de entrega (por defecto) o, con USE_ZIP = True,
# usa un paquete generado con `git archive` (contractrisk_m3_final_codigo.zip).
USE_ZIP = False
REPO = 'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git'
BRANCH = 'Entrega/contractrisk-m3'
ROOT = Path('/content/contractrisk_m3_final')
os.chdir('/content')  # el directorio actual puede no existir si la celda se re-ejecutó tras borrar ROOT
if ROOT.exists():
    raise RuntimeError('Use una sesión nueva: la carpeta ya existe y la evidencia anterior no se sobrescribe')
if USE_ZIP:
    from google.colab import files
    uploaded = files.upload()  # se guarda en /content
    if len(uploaded) != 1:
        raise ValueError('Suba únicamente contractrisk_m3_final_codigo.zip')
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(Path('/content') / next(iter(uploaded))) as z:
        for member in z.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Ruta no permitida en ZIP')
        z.extractall(ROOT)
    COMMIT = z.comment.decode() or 'zip-sin-commit'
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
    COMMIT = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
os.chdir(ROOT)
print('Código evaluado:', COMMIT)
assert Path('m3/run_final_m3.py').is_file(), 'El código no contiene la corrida final: use la rama o ZIP correctos'

In [ ]:
def run_visible(command, log_path):
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f'Comando falló ({code}); consulte {log_path}')

run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], 'installation.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests.log')
run_visible([sys.executable, '-c', "import sys; sys.path.insert(0,'m3'); from sqlite_support import ensure_sqlite; print(ensure_sqlite()); import torch; assert torch.cuda.is_available(), 'Active GPU T4'; print(torch.cuda.get_device_name(0))"], 'environment.log')

In [ ]:
# Ejecuta los seis sistemas, el juez y RAGAS; escribe todo en una carpeta nueva.
run_visible([sys.executable, '-u', 'm3/run_final_m3.py', '--output', 'final_m3_resultados'], 'execution.log')

In [ ]:
# Tablas recalculadas desde el detalle por consulta.
from IPython.display import Markdown, display
display(Markdown(Path('final_m3_resultados/tablas_m3.md').read_text(encoding='utf-8')))

In [ ]:
# Trazas de la herramienta en el sistema final: decisiones reales del agente por pregunta.
import json
for line in Path('final_m3_resultados/harness_rows.jsonl').read_text(encoding='utf-8').splitlines():
    item = json.loads(line)
    row, rec = item['row'], item['record']
    if row['variant'] != 'sistema_final':
        continue
    decisions = [(e['step'], (e['decision'] or {}).get('action'), (e['tool_result'] or {}).get('ok'),
                  (e['tool_result'] or {}).get('operation') or (e['tool_result'] or {}).get('error'))
                 for e in rec.get('tool_trace', [])]
    out = rec.get('output') or {}
    print(row['eval_id'], row['kind'], '| llamadas:', rec.get('tool_calls'), '|', rec.get('termination'), '|', decisions)
    print('   respuesta:', out.get('respuesta'), '| abstencion:', out.get('abstencion'), '| dominio:', row['domain_pass'], row['domain_reasons'])

In [ ]:
# Empaqueta resultados, logs y commit evaluado en un ZIP descargable.
import shutil
from google.colab import files
result_dir = ROOT / 'final_m3_resultados'
result_dir.mkdir(exist_ok=True)
(result_dir / 'commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, result_dir / log.name)
meta = result_dir / 'metadata.json'
print(meta.read_text(encoding='utf-8')[:3000] if meta.exists() else 'No se creó metadata: revisar logs.')
bundle = shutil.make_archive(str(ROOT / 'contractrisk_m3_final_resultados'), 'zip', result_dir)
files.download(bundle)

## Resultados

La carpeta `final_m3_resultados` contiene las respuestas, contextos y trazas por consulta (`harness_rows.jsonl`), el scorecard (`scorecard_m3.csv`), RAGAS (`ragas_m3.csv`), el detalle por caso, las tablas (`tablas_m3.md`) y los metadatos de la corrida (modelos, versiones y hashes). La última celda empaqueta esta carpeta, junto con los logs y el commit evaluado, en `contractrisk_m3_final_resultados.zip`. La corrida usada en el reporte está en `m3/results/final_m3/`.